# 04 - Strategy backtests

**What:** backtest the enabled strategies with realistic costs, ADV caps, and a harsher-costs stress run, then apply the claims-discipline gate.

**Why:** an event-study edge is worthless if it doesn't survive spread + impact in thin names. We only call something alpha if it clears every automatable check.

**Discipline:** survives costs AND harsher slippage; no single trade dominates; >1 event; bootstrap CI lower bound > 0; enough liquidity; OOS not negative.

In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import pandas as pd
from index_flow.config import load_config
from index_flow.event_builder import build_and_enrich
from index_flow.backtester import run_all_strategies
from index_flow.diagnostics import compute_metrics, claims_gate, breakdown
cfg = load_config(); cfg.ensure_dirs()
events, _ = build_and_enrich(cfg)
ledger = run_all_strategies(cfg, events)
print('ledger rows:', len(ledger))

In [ ]:
if ledger.empty:
    print('No trades. Ingest data and ensure events pass tradeability filters.')
else:
    print('Overall:', compute_metrics(ledger))
    print('\nClaims gate:', claims_gate(ledger))
    for dim in ['strategy','theme','provider','flow_pressure_bucket','liquidity_bucket']:
        bd = breakdown(ledger, events, dim)
        if not bd.empty:
            print('\nBy', dim); display(bd)